In [1]:

import os
import json


with open(os.path.join('../', 'data', 'cossim_bw_categories', 'category_similarity_None.json'), 'r') as f:
    category_similarity = json.load(f)

category_similarity["classification"]['spacecraft']["far"][0][0]



'hockey team'

## 文を区切るpatternのテスト

In [1]:
import re

text = "これはテストです。これは2文目です！これは3文目です？これは4文目です。"
parts = re.split(r'(?<=[。．!?！？])\s*', text)
for part in parts:
    print(f"part: '{part}'")

part: 'これはテストです。'
part: 'これは2文目です！'
part: 'これは3文目です？'
part: 'これは4文目です。'
part: ''


## torch.tensor(l) を 2で割るときの挙動をテスト

In [2]:

import torch

l = [
    [0, 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9, 10, 11]
]
l = torch.tensor(l)
l // 2

tensor([[0, 0, 1, 1],
        [2, 2, 3, 3],
        [4, 4, 5, 5]])

In [3]:
b = 1
attention_mask = torch.tensor([
    [[1, 1, 1, 1, 1, 1, 0], 
    [1, 1, 1, 1, 0, 0, 0], 
    [1, 1, 0, 0, 0, 0, 0]],
    
    [[1, 1, 1, 1, 1, 1, 0], 
    [1, 1, 1, 1, 0, 0, 0], 
    [1, 1, 0, 0, 0, 0, 0]]
])
seq_len = attention_mask[b].sum().item()
half_seq_len = seq_len // 2
print(f"seq_len: {seq_len}, half_seq_len: {half_seq_len}")

seq_len: 12, half_seq_len: 6


## modelの隠れ層3層分について、複数トークン位置の隠れ状態を平均し、さらに3層分の平均を取るときのコードをテスト

In [4]:

import os
import sys
import random
import re
import json

import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

/home/toko/project/EmbedNewConcept-20260305/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
model_version = 3
model_size = 1

model_name = f"google/gemma-{model_version}-{model_size}b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading weights: 100%|██████████| 340/340 [00:04<00:00, 72.94it/s]


In [71]:
batch_texts = [
    "これは別のテキストです。これも複数の文からなっています！例えば、これは2文目です？そして、これは3文目です。",
    "これはテストです。これは2文目です！これは3文目です？これは4文目です。",
]
batch_texts = [text + tokenizer.eos_token for text in batch_texts]


inputs = tokenizer(
    batch_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    add_special_tokens=False
).to(model.device)

input_ids = inputs.input_ids
attention_mask = inputs.attention_mask


print(inputs)

{'input_ids': tensor([[ 49834,  95466,  95830,   3652, 236924, 185074, 120270, 237364,   5077,
         237050,  32881, 237354,  76410, 236951,  49834, 236778, 237364, 237472,
           3652, 237536,  29773, 236951,  49834, 236800, 237364, 237472,   3652,
         236924,      1],
        [     0,      0,      0,      0,      0,      0,  49834,  88733,   3652,
         236924,  49834, 236778, 237364, 237472,   3652, 237354,  49834, 236800,
         237364, 237472,   3652, 237536,  49834, 236812, 237364, 237472,   3652,
         236924,      1]], device='cuda:5'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]], device='cuda:5')}


In [96]:
mix_layers = True
layer_index = 4
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    all_hs = outputs.hidden_states


# *** 前後3層mixでterm_vecを作る場合 ***
if mix_layers:
    mixed_layers = [3,4,5]  # mixする層のインデックスを指定（例: 3,4,5層をmix）
    # ** 前後3層の隠れ状態を平均する:
    target_layer_hs = torch.stack(
        [all_hs[lid] for lid in mixed_layers],
        dim=0
    )  # 指定層の出力 [3, batch_size, seq_len, d]
else:
    target_layer_hs = all_hs[layer_index].unsqueeze(0)      # (1, batch_size, seq_len, d)

print(target_layer_hs.shape)

vecs = []
for t_idx in range(target_layer_hs.shape[1]):

    # 1 が立っている位置を取得 [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1] -> valid_pos = [6, 7, 8, 9, 10, 11] 
    valid_pos = torch.nonzero(attention_mask[t_idx], as_tuple=False).squeeze(-1)
    print(f"valid_pos for batch {t_idx}: {valid_pos}")
    if valid_pos.numel() == 0:
        # 全部 padding の場合
        pos_begin = 0
        pos_end = 0
    else:
        pos_begin = valid_pos[0].item()
        pos_end = valid_pos[-1].item() + 1   # slice用に end は exclusive


    # pos_begin_second_sent = (pos_begin + pos_end) // 2 # == pos_begin + (pos_end - pos_begin) / 2
    # pos_begin = pos_begin_second_sent

    print(f'({pos_begin}, {pos_end})')

    i = 0
    # vec = target_layer_hs[:, t_idx, pos_begin:pos_end, :].mean(dim=0).mean(dim=1)  # (mix層数, 1, max_token数, d) -> (batch_size, d) 前後3層を平均した隠れ状態のうち、valid_tokenの部分を平均する
    print(i, target_layer_hs.shape)         # [3, 2, 29, 1152]) --- (mix層数, batch_size, seq_len, d)
    i += 1
    print(i, target_layer_hs[:].shape)          # [3, 2, 29, 1152])
    i += 1
    print(i, target_layer_hs[:, t_idx].shape)   # [3, 29, 1152])
    i += 1
    print(i, target_layer_hs[:, t_idx, pos_begin:pos_end].shape)    # [3, 23, 1152]
    i += 1
    print(i, target_layer_hs[:, t_idx, pos_begin:pos_end, ].shape) # [3, 23, 1152]
    i += 1
    print(i, target_layer_hs[:, t_idx, pos_begin:pos_end, :].shape) # [3, 23, 1152]
    i += 1
    print(i, target_layer_hs[:, t_idx, pos_begin:pos_end, :].mean(dim=0).shape) # [23, 1152]
    i += 1
    print(i, target_layer_hs[:, t_idx, pos_begin:pos_end, :].mean(dim=0).mean(dim=0).shape) # [23])
    
    vec = target_layer_hs[:, t_idx, pos_begin:pos_end, :].mean(dim=0).mean(dim=0)  # (mix層数, valid_token数, d) -> (valid_token数, d) 前後3層を平均した隠れ状態のうち、valid_tokenの部分を平均する
    print('vec:', vec.shape)
    vecs.append(vec.detach().cpu())
vecs = torch.stack(vecs, dim=0)  # (batch_size, d)

torch.Size([3, 2, 29, 1152])
valid_pos for batch 0: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28], device='cuda:5')
(0, 29)
0 torch.Size([3, 2, 29, 1152])
1 torch.Size([3, 2, 29, 1152])
2 torch.Size([3, 29, 1152])
3 torch.Size([3, 29, 1152])
4 torch.Size([3, 29, 1152])
5 torch.Size([3, 29, 1152])
6 torch.Size([29, 1152])
7 torch.Size([1152])
vec: torch.Size([1152])
valid_pos for batch 1: tensor([ 6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
        24, 25, 26, 27, 28], device='cuda:5')
(6, 29)
0 torch.Size([3, 2, 29, 1152])
1 torch.Size([3, 2, 29, 1152])
2 torch.Size([3, 29, 1152])
3 torch.Size([3, 23, 1152])
4 torch.Size([3, 23, 1152])
5 torch.Size([3, 23, 1152])
6 torch.Size([23, 1152])
7 torch.Size([1152])
vec: torch.Size([1152])


In [101]:

print('vecsを1つずつ足す場合:')
sum_vec = torch.zeros_like(vec).cpu()  # (d,) ... E[0]と同じshapeとdtypeのゼロベクトルを作成
print(sum_vec.shape)  # (d,)
for vec in vecs:
    print(sum_vec.sum().item(), '+', vec.sum().item())  # 0.0
    sum_vec += vec  # (d,) ... vecをsum_vecに加算
    print(sum_vec.sum().item(), 'shape: ', sum_vec.shape)  # (d,)

# ***
print('\n\n一気にvecs.sum()で足す場合:')
sum_vec = torch.zeros_like(vec).cpu()  # (d,) ... E[0]と同じshapeとdtypeのゼロベクトルを作成
print(sum_vec.shape)  # (d,)

print(sum_vec.sum().item(), '+', vecs[0].sum().item(), '+', vecs[1].sum().item())  # 0.0
sum_vec += vecs.sum(dim=0)  # (d,) ... vecsの全ベクトルを足し合わせる
print(sum_vec.sum().item(), 'shape: ', sum_vec.shape)  # (d,)




vecsを1つずつ足す場合:
torch.Size([1152])
0.0 + -116.1875
-116.1875 shape:  torch.Size([1152])
-116.1875 + -183.5
-299.25 shape:  torch.Size([1152])


一気にvecs.sum()で足す場合:
torch.Size([1152])
0.0 + -116.1875 + -183.5
-299.25 shape:  torch.Size([1152])


In [20]:

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
print(f"input_ids shape: {input_ids.shape}, attention_mask shape: {attention_mask.shape}")
print(f"input_ids: {input_ids}")
print(f"attention_mask: {attention_mask}")

input_ids shape: torch.Size([2, 29]), attention_mask shape: torch.Size([2, 29])
input_ids: tensor([[ 49834,  95466,  95830,   3652, 236924, 185074, 120270, 237364,   5077,
         237050,  32881, 237354,  76410, 236951,  49834, 236778, 237364, 237472,
           3652, 237536,  29773, 236951,  49834, 236800, 237364, 237472,   3652,
         236924,      1],
        [     0,      0,      0,      0,      0,      0,  49834,  88733,   3652,
         236924,  49834, 236778, 237364, 237472,   3652, 237354,  49834, 236800,
         237364, 237472,   3652, 237536,  49834, 236812, 237364, 237472,   3652,
         236924,      1]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])


### pool_hs_typeによって、平均対象のtoken位置を決めるコードのテスト

In [21]:
# batch内の、各文が始まる位置と終わる位置を求める
for i in range(len(batch_texts)):
    input_id = input_ids[i]
    attention_mask_i = attention_mask[i]
    seq_len = attention_mask_i.sum().item()
    print(f"seq_len: {seq_len}")
    
    # 文の区切りは、eos_token_idであると仮定する
    eos_token_id = tokenizer.eos_token_id
    sentence_boundaries = (input_id == eos_token_id).nonzero(as_tuple=True)[0].tolist()
    
    # 文の開始位置と終了位置を求める
    sentence_positions = []
    start_pos = 0
    for end_pos in sentence_boundaries:
        sentence_positions.append((start_pos, end_pos.item()))
        start_pos = end_pos.item() + 1
    
    print(f"sentence_positions: {sentence_positions}")


seq_len: 29


AttributeError: 'int' object has no attribute 'item'

In [23]:

inputs = tokenizer(
    batch_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

input_ids = inputs["input_ids"]         # (B, T)
attention_mask = inputs["attention_mask"] # (B, T)


batch_size, seq_len = attention_mask.shape
sentence_vecs = []
token_spans = []

for b in range(batch_size):
    mask = attention_mask[b]  # (T,)

    # 1 が立っている位置を取得
    valid_pos = torch.nonzero(mask, as_tuple=False).squeeze(-1)
    print(f"valid_pos for batch {b}: {valid_pos}")

    if valid_pos.numel() == 0:
        # 全部 padding の場合
        pos_begin = 0
        pos_end = 0
        # vec = torch.zeros(layer_hs.size(-1), device=layer_hs.device, dtype=layer_hs.dtype)
    else:
        pos_begin = valid_pos[0].item()
        pos_end = valid_pos[-1].item() + 1   # slice用に end は exclusive

        # 文の token 範囲だけ平均
        # vec = layer_hs[b, pos_begin:pos_end, :].mean(dim=0)  # (H,)

    token_spans.append((pos_begin, pos_end))
    # sentence_vecs.append(vec)

# sentence_vecs = torch.stack(sentence_vecs, dim=0)  # (B, H)

print("token spans:", token_spans)
# print("sentence_vecs shape:", sentence_vecs.shape)

valid_pos for batch 0: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])
valid_pos for batch 1: tensor([ 6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
        24, 25, 26, 27, 28, 29])
token spans: [(0, 30), (6, 30)]


In [ ]:
print(valid_pos)
print(pos_begin, pos_end)
pos_begin_after_half = (pos_begin + pos_end) // 2 # == pos_begin + (pos_end - pos_begin) / 2
print(f"after half: {pos_begin_after_half}")


tensor([ 6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
        24, 25, 26, 27, 28, 29])
6 30
after half: 18


In [26]:
(pos_end - pos_begin) / 2 + pos_begin

18.0

In [46]:
batch_texts = [
    "これはテストです。あいうえお かきくけこ。",
    "これは別のテキストです。",
]

In [49]:
pool_hs_type = "mean_pool"
data_type = "wiki_summary_repeat"

if data_type == "wiki_summary_repeat":
    batch_texts = [
        t * 2 for t in batch_texts
    ]


if pool_hs_type == "eos":
    # EOS を明示的に末尾へ追加
    batch_texts = [text + tokenizer.eos_token for text in batch_texts]

inputs = tokenizer(
    batch_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    add_special_tokens=False
)




input_ids = inputs["input_ids"]         # (B, T)
attention_mask = inputs["attention_mask"] # (B, T)

for i in range(len(batch_texts)):
    print(f"batch_texts[{i}]: {batch_texts[i]}")
    print(f"input_ids[{i}]: {input_ids[i]}")

if pool_hs_type == "eos":
    # 各系列について EOS token の最後の出現位置を取る
    eos_mask = (input_ids == tokenizer.eos_token_id)
    
for t_idx in range(input_ids.size(0)):

    # 1 が立っている位置を取得 [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1] -> valid_pos = [6, 7, 8, 9, 10, 11] 
    valid_pos = torch.nonzero(attention_mask[t_idx], as_tuple=False).squeeze(-1)
    # print(f"valid_pos for batch {t_idx}: {valid_pos}")
    if valid_pos.numel() == 0:
        # 全部 padding の場合
        pos_begin = 0
        pos_end = 0
    else:
        pos_begin = valid_pos[0].item()
        pos_end = valid_pos[-1].item() + 1   # slice用に end は exclusive


    if pool_hs_type == "eos":
        eos_positions = torch.where(eos_mask[t_idx])[0]
        if len(eos_positions) == 0:
            raise ValueError(f"EOS token が見つかりません: {batch_texts[t_idx]}")
        eos_pos = eos_positions[-1].item()
        pos_begin = eos_pos
        pos_end = eos_pos + 1
        # vec = layer_hs[t_idx, eos_pos, :]   # (H,)
            

    elif pool_hs_type == "last_token":
        seq_len = attention_mask[t_idx].sum().item()
        # vec = layer_hs[t_idx, seq_len - 1, :]  # (H,)
        pos_begin = pos_end - 1


    elif pool_hs_type == "mean_pool":
        # seq_len = attention_mask[t_idx].sum().item()

        if data_type == "wiki_summary_repeat":
            # wiki summaryを繰り返してプロンプトとする場合は、2回目の文のみの隠れ状態を平均する
            # seq_first_half_len = seq_len // 2   # 1文が5tokens → id:0,1,2,3,4 が1文目、id:5,6,7,8,9 が2文目の場合、seq_len=10, seq_first_half_len=5 となる
            pos_begin_second_sent = (pos_begin + pos_end) // 2 # == pos_begin + (pos_end - pos_begin) / 2
            pos_begin = pos_begin_second_sent
            # vec = layer_hs[t_idx, seq_first_half_len:seq_len, :].mean(dim=0)  # (H,)
        # else:
        #     # vec = layer_hs[t_idx, :seq_len, :].mean(dim=0)  # (H,)

    else:
        raise ValueError(f"Unknown pool_hs_type: {pool_hs_type}")
    
    # vec = layer_hs[t_idx, pos_begin:pos_end, :].mean(dim=0)  # (H,)
    print(f"pos_begin: {pos_begin}, pos_end: {pos_end}")
    print(f"\tattention_mask: {attention_mask[t_idx]},\n\t valid_pos: {valid_pos}, \n\t valid part in batch_text: {input_ids[t_idx][pos_begin:pos_end]}")


batch_texts[0]: これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。これはテストです。あいうえお かきくけこ。
input_ids[0]: tensor([ 49834,  88733,   3652, 236924, 237268,   8591, 237495, 237328,  18910,
        237264, 237234, 237406, 237171, 236924,  49834,  88733,   3652, 236924,
        237268,   8591, 237495, 237328,  18910, 237264, 237234, 237406, 237171,
        236924,  49834,  88733,   3652, 236924, 237268,   8591, 237495, 237328,
         18910, 237264, 237234, 237406, 237171, 236924,  49834,  88733,   3652,
        236924, 237268,   8591, 237495, 237328,  18910, 237264, 237234, 237406,
        237171, 236924,  49834,  88733,   3652, 236924, 237268,   8591, 237495,
        237328,  18910, 237264, 237234, 237406, 237171, 236924,  49834,  88733,
          3652, 236924, 237268,   8591, 237495, 237328,  18910, 237264, 237234,
        237406, 237171, 236924,  49834,  88733,   3652, 236924, 237268,   8591,
 

In [33]:
(6 + 30) / 2

18.0

In [13]:
eos_mask = (input_ids == tokenizer.eos_token_id)
eos_mask

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False,  True]])

In [14]:
eos_positions = torch.where(eos_mask[b])[0]
eos_positions

tensor([29])

In [ ]:


texts = [
    "これは短い文です。",
    "これは少し長めの文です。"
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

input_ids = inputs["input_ids"].to(model.device)           # (B, T)
attention_mask = inputs["attention_mask"].to(model.device) # (B, T)

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
    )

# 例: 最終層の hidden state を使う
layer_hs = outputs.hidden_states[-1]   # (B, T, H)

batch_size, seq_len = attention_mask.shape
sentence_vecs = []
token_spans = []

for b in range(batch_size):
    mask = attention_mask[b]  # (T,)

    # 1 が立っている位置を取得
    valid_pos = torch.nonzero(mask, as_tuple=False).squeeze(-1)

    if valid_pos.numel() == 0:
        # 全部 padding の場合
        pos_begin = 0
        pos_end = 0
        vec = torch.zeros(layer_hs.size(-1), device=layer_hs.device, dtype=layer_hs.dtype)
    else:
        pos_begin = valid_pos[0].item()
        pos_end = valid_pos[-1].item() + 1   # slice用に end は exclusive

        # 文の token 範囲だけ平均
        vec = layer_hs[b, pos_begin:pos_end, :].mean(dim=0)  # (H,)

    token_spans.append((pos_begin, pos_end))
    sentence_vecs.append(vec)

sentence_vecs = torch.stack(sentence_vecs, dim=0)  # (B, H)

print("token spans:", token_spans)
print("sentence_vecs shape:", sentence_vecs.shape)

## 言語リスト

In [1]:
import pycountry

iso639_1_languages = sorted(
    [
        {"code": lang.alpha_2, "name": lang.name}
        for lang in pycountry.languages
        if hasattr(lang, "alpha_2") and hasattr(lang, "name")
    ],
    key=lambda x: x["code"]
)

print(iso639_1_languages[:10])
print(len(iso639_1_languages))

[{'code': 'aa', 'name': 'Afar'}, {'code': 'ab', 'name': 'Abkhazian'}, {'code': 'ae', 'name': 'Avestan'}, {'code': 'af', 'name': 'Afrikaans'}, {'code': 'ak', 'name': 'Akan'}, {'code': 'am', 'name': 'Amharic'}, {'code': 'an', 'name': 'Aragonese'}, {'code': 'ar', 'name': 'Arabic'}, {'code': 'as', 'name': 'Assamese'}, {'code': 'av', 'name': 'Avaric'}]
184


In [6]:
iso639_1_languages[100:]

[{'code': 'mh', 'name': 'Marshallese'},
 {'code': 'mi', 'name': 'Maori'},
 {'code': 'mk', 'name': 'Macedonian'},
 {'code': 'ml', 'name': 'Malayalam'},
 {'code': 'mn', 'name': 'Mongolian'},
 {'code': 'mr', 'name': 'Marathi'},
 {'code': 'ms', 'name': 'Malay (macrolanguage)'},
 {'code': 'mt', 'name': 'Maltese'},
 {'code': 'my', 'name': 'Burmese'},
 {'code': 'na', 'name': 'Nauru'},
 {'code': 'nb', 'name': 'Norwegian Bokmål'},
 {'code': 'nd', 'name': 'North Ndebele'},
 {'code': 'ne', 'name': 'Nepali (macrolanguage)'},
 {'code': 'ng', 'name': 'Ndonga'},
 {'code': 'nl', 'name': 'Dutch'},
 {'code': 'nn', 'name': 'Norwegian Nynorsk'},
 {'code': 'no', 'name': 'Norwegian'},
 {'code': 'nr', 'name': 'South Ndebele'},
 {'code': 'nv', 'name': 'Navajo'},
 {'code': 'ny', 'name': 'Chichewa'},
 {'code': 'oc', 'name': 'Occitan (post 1500)'},
 {'code': 'oj', 'name': 'Ojibwa'},
 {'code': 'om', 'name': 'Oromo'},
 {'code': 'or', 'name': 'Oriya (macrolanguage)'},
 {'code': 'os', 'name': 'Ossetian'},
 {'code': 

In [2]:
langs = [lang['name'] for lang in iso639_1_languages]
print(langs[:10])

['Afar', 'Abkhazian', 'Avestan', 'Afrikaans', 'Akan', 'Amharic', 'Aragonese', 'Arabic', 'Assamese', 'Avaric']


In [5]:
import re

# langs内の言語に該当するか、大文字小文字関係なく完全一致するかで判断する正規表現パターンを作成
lang_pattern = re.compile(r'\b(' + '|'.join(re.escape(lang) for lang in langs) + r')\b', re.IGNORECASE)

examples = [
    'English',
    'english',
    'This is English.', # no match
]

for word in examples:
    # 完全一致で判断
    if lang_pattern.fullmatch(word.strip()):
        print(f"'{word}' matches a language in the list.")
    else:
        print(f"'{word}' does NOT match any language in the list.")

'English' matches a language in the list.
'english' matches a language in the list.
'This is English.' does NOT match any language in the list.


## 国名・都市名リスト

In [6]:
import pycountry

country_names = sorted({
    country.name
    for country in pycountry.countries
    if hasattr(country, "name") and country.name
})

print(country_names)

['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia, Plurinational State of', 'Bonaire, Sint Eustatius and Saba', 'Bosnia and Herzegovina', 'Botswana', 'Bouvet Island', 'Brazil', 'British Indian Ocean Territory', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China', 'Christmas Island', 'Cocos (Keeling) Islands', 'Colombia', 'Comoros', 'Congo', 'Congo, The Democratic Republic of the', 'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czechia', "Côte d'Ivoire", 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', '

In [ ]:
import geonamescache

gc = geonamescache.GeonamesCache(min_city_population=15000)

city_names = sorted({
    city["name"]
    for city in gc.get_cities().values()
    if city.get("name")
})

print(city_names)

FileNotFoundError: [Errno 2] No such file or directory: '/home/toko/project/EmbedNewConcept-20260305/.venv/lib/python3.12/site-packages/geonamescache/data/cities20000.json'